# Export heatwave metrics
- This script is used to export heatwave metric results.
- Simulations: CNTL, TranAlbe

In [1]:
import pandas as pd
import numpy as np
region='HK'

In [2]:
for threshold in [33, 35]:
    for scenario in ['cntl', 'tran_albe']:
        df_temp = pd.read_csv(f'../temperature_trend/data_for_figure/{region}_tsa_{scenario}_urban_rural.csv') 
        df_temp['time'] = pd.to_datetime(df_temp['time'])
        df_temp['date'] = df_temp['time'].dt.date
        heatwave_event_list = []
        for var in ['TSA', 'TSA_U', 'TSA_R']:
            df_temp_tsau = df_temp[['date', var]].copy()
            df_tmax_tsau = df_temp_tsau.groupby('date').max().reset_index()
            df_tmax_tsau['is_hot'] = df_tmax_tsau[var] >= threshold
            # Identify consecutive groups
            df_tmax_tsau['group'] = (df_tmax_tsau['is_hot'] != df_tmax_tsau['is_hot'].shift()).cumsum()

            # For each group, count how many consecutive days satisfy TSA > 33
            heatwave_groups = df_tmax_tsau.groupby('group').filter(lambda x: x['is_hot'].all() and len(x) >= 3)

            # Optionally, extract the start and end dates of each heatwave event
            heatwave_events = (df_tmax_tsau.groupby('group').filter(lambda x: x['is_hot'].all() and len(x) >= 3)
                               .groupby('group').agg(start=('date', 'first'), end=('date', 'last'), duration=('date', 'count'))
                               .reset_index(drop=True)
                               )
            heatwave_events['var'] = var
            heatwave_event_list.append(heatwave_events)
        df_heatwave_event_all = pd.concat(heatwave_event_list, ignore_index=True)    
        output_filename = f'./data_for_figure/{region}_heatwave_events_{threshold}_{scenario}_urban_rural.csv' #_no_sst
        df_heatwave_event_all.to_csv(output_filename, index=False) 
        df_heatwave_event_all.head()

In [3]:
def heatwave_frequency(df_events):
    frequency = df_events.groupby('year').size().reset_index(name='count')
    return frequency

def heatwave_seasonal_length(df_events): 
    yearly_first_heatwave = df_events.groupby('year')['start'].min().reset_index()
    yearly_last_heatwave = df_events.groupby('year')['end'].max().reset_index()
    yearly_heatwave = pd.merge(yearly_first_heatwave, yearly_last_heatwave, on='year')
    yearly_heatwave['season_length'] = (yearly_heatwave['end'] - yearly_heatwave['start']).dt.days
    return yearly_heatwave[['year', 'season_length']]

def heatwave_intensity(df_events, df_temperature):
    yearly_heatwaves = df_events.groupby('year')
    intensity = yearly_heatwaves.apply(
        lambda x: df_temperature.loc[(df_temperature['date'] >= x['start'].min()) & 
                              (df_temperature['date'] <= x['end'].max()), var].mean()).reset_index(name='intensity')
    return intensity

def heatwave_duration(df_events):
    duration = df_events.groupby('year')['duration'].mean().reset_index()
    return duration

In [4]:
var = 'TSA_U'
metric_list = []
for threshold in [33, 35]:
    metric_results = []
    for scenario in ['cntl', 'tran_albe']:
        df_heatwave_events = pd.read_csv(f'data_for_figure/{region}_heatwave_events_{threshold}_{scenario}_urban_rural.csv') #_no_sst
        df_heatwave_events_urban = df_heatwave_events[df_heatwave_events['var']==var].copy()
        df_heatwave_events_urban['start'] = pd.to_datetime(df_heatwave_events_urban['start'])
        df_heatwave_events_urban['end'] = pd.to_datetime(df_heatwave_events_urban['end'])
        df_heatwave_events_urban['year'] = df_heatwave_events_urban['start'].dt.year
        heatwave_metrics = []
        heatwave_metrics.append(heatwave_frequency(df_heatwave_events_urban).set_index('year'))
        heatwave_metrics.append(heatwave_seasonal_length(df_heatwave_events_urban).set_index('year'))
        df_temp = pd.read_csv(f'../temperature_trend/data_for_figure/{region}_tsa_{scenario}_urban_rural.csv') #_no_sst
        df_temp['time'] = pd.to_datetime(df_temp['time'])
        df_temp['date'] = df_temp['time'].dt.date
        df_temp_tsau = df_temp[['date', var]].copy()
        df_tmax_tsau = df_temp_tsau.groupby('date').max().reset_index()
        df_tmax_tsau['is_hot'] = df_tmax_tsau[var] >= threshold
        df_temp_urban = df_tmax_tsau[df_tmax_tsau['is_hot']].copy()
        df_temp_urban['date'] = pd.to_datetime(df_temp_urban['date'])
        df_temp_urban['year'] = df_temp_urban['date'].dt.year
        heatwave_metrics.append(heatwave_intensity(df_heatwave_events_urban, df_temp_urban).set_index('year'))
        heatwave_metrics.append(heatwave_duration(df_heatwave_events_urban).set_index('year'))
        df_heatwave_metrics_urban = pd.concat(heatwave_metrics, axis=1)
        metric_results.append(df_heatwave_metrics_urban)
    df_heatwave_metrics_urban_all = pd.concat(metric_results, keys=['cntl', 'tran_albe'], names=['scenario', 'year']).reset_index()  
    df_heatwave_metrics_urban_all['totoal_days'] = (df_heatwave_metrics_urban_all['count'] * df_heatwave_metrics_urban_all['duration'])
    df_heatwave_metrics_urban_all['exposure'] = (df_heatwave_metrics_urban_all['count'] * df_heatwave_metrics_urban_all['duration']) * df_heatwave_metrics_urban_all['intensity']
    df_heatwave_metrics_urban_all = np.round(df_heatwave_metrics_urban_all, 1)
    df_heatwave_metrics_urban_all['threshold'] = threshold
    metric_list.append(df_heatwave_metrics_urban_all)
df_metric = pd.concat(metric_list, axis=0, ignore_index=True)
df_metric

/tmp/ipykernel_1390941/523808768.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  intensity = yearly_heatwaves.apply(
/tmp/ipykernel_1390941/523808768.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  intensity = yearly_heatwaves.apply(
/tmp/ipykernel_1390941/523808768.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future vers

,scenario,year,count,season_length,intensity,duration,totoal_days,exposure,threshold
0,cntl,2035,9,168,34.9,16.1,145.0,5063.5,33
1,cntl,2036,9,164,35.0,13.8,124.0,4341.9,33
2,cntl,2037,6,121,35.1,18.5,111.0,3890.9,33
3,cntl,2038,9,167,35.3,13.8,124.0,4371.2,33
4,cntl,2039,5,128,35.3,23.6,118.0,4168.0,33
5,tran_albe,2035,9,168,34.9,16.0,144.0,5022.6,33
6,tran_albe,2036,9,164,34.9,13.8,124.0,4332.7,33
7,tran_albe,2037,7,120,34.9,15.6,109.0,3806.1,33
8,tran_albe,2038,7,130,35.2,16.6,116.0,4088.7,33
9,tran_albe,2039,4,120,35.1,28.5,114.0,3998.0,33


In [ ]:
# total heatwave days in 2039, threshold: 35°C
np.round((53-45)/53*100,1)

15.1

In [6]:
# season length of heatwave in 2039, threshold: 35°C
np.round((117-82)/117*100,1)

29.9